# download packages and data

In [4]:
#NLTK imports
import nltk
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.corpus import stopwords
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('omw-1.4')

#other imports
import pandas as pd
import numpy as np
import ast
import re
import statistics

#statistics
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr, pearsonr

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lucia\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lucia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lucia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\lucia\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [5]:
from nltk.stem import WordNetLemmatizer as wnl
from nltk.corpus import wordnet as wn
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')

from nltk.corpus import wordnet_ic
nltk.download('wordnet_ic')
brown_ic = wordnet_ic.ic('ic-brown.dat')


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\lucia\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\lucia\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\universal_tagset.zip.
[nltk_data] Downloading package wordnet_ic to
[nltk_data]     C:\Users\lucia\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\wordnet_ic.zip.


In [27]:
#load data (words and definitions) from which metrics for BM need to be extracted
df = pd.read_csv('..\data\EC.csv')

In [18]:
#load psycholinguistic norm datasets
concreteness_df = pd.read_excel('..\data\psycholinguistic norms\Brisbaert_40K_Concreteness.xlsx')
physicality_df = pd.read_excel('..\data\psycholinguistic norms\SER_EN_5K.xls')
glasgow_df = pd.read_csv('..\data\psycholinguistic norms\glasgow_norms.csv')
mrc_c = pd.read_csv('..\data\psycholinguistic norms\MRC_corpus.csv')

glasgow_df = glasgow_df[1:]
keep_columns = ['Words', 'AROU', 'VAL', 'DOM', 'IMAG', 'FAM']
glasgow_df = glasgow_df[keep_columns]
glasgow_df = glasgow_df.astype({'AROU': float, 'VAL': float, 'DOM': float, 'IMAG': float, 'FAM': float})



In [20]:
#download static models (this takes some time)
import gensim.downloader as api
nb_17 = api.load("conceptnet-numberbatch-17-06-300")
nb17_vocab = list(nb_17.key_to_index.keys())
nb17_vocab_en = [i.split('/')[3] for i in nb17_vocab if i.startswith('/c/en/')]

w2v = api.load('word2vec-google-news-300')
w2v_vocab = list(w2v.key_to_index.keys())

[==================================================] 100.0% 1168.7/1168.7MB downloaded
[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
#execute if only words and no definitions are provided
df['synsets'] = df['Word'].apply(lambda x: [wordnet.synsets(x) if type(x) != float else None])
df = df.explode('synsets')
df.reset_index(drop=True, inplace=True)

for i, row in df.iterrows():
  try:
    df.at[i, 'definitions'] = row['synsets'].definition()
  except:
    df.at[i, 'definitions'] = None
  try:
    df.at[i, 'examples'] = list(row['synsets'].examples())
  except:
    df.at[i, 'examples'] = None
  try:
    df.at[i, 'synonyms'] = [str(lemma.name()) for lemma in row['synsets'].lemmas()]
  except:
    df.at[i, 'synonyms'] = None

In [28]:
df_vocab = []
for i, row in df.iterrows():
  df_vocab.append(row['Word'])
  try:
    for w in word_tokenize(row['definitions']):
      df_vocab.append(w)
  except: pass
print(len(df_vocab))
df_vocab = set(df_vocab)
print(len(df_vocab))
df_vocab = list(df_vocab)

8313
2141


# extend psycholinguistic norms

In [29]:
#extend psycholinguistic features to cover oov
psycholinguistic_norms = pd.DataFrame(columns=['word', 'concreteness', 'physicality', 'imageability', 'familiarity', 'dataset'])
psycholinguistic_norms['word'] = df_vocab
for i, row in psycholinguistic_norms.iterrows():
  if row['word'] in concreteness_df['Word'].values:
    psycholinguistic_norms.at[i, 'dataset'] = 'concreteness'
    psycholinguistic_norms.at[i, 'concreteness'] = concreteness_df.loc[concreteness_df['Word'] == row['word'], 'Conc.M'].values[0]
  else:
    psycholinguistic_norms.at[i, 'concreteness'] = None

  if row['word'] in physicality_df['Word'].values:
    psycholinguistic_norms.at[i, 'dataset'] = 'physicality'
    psycholinguistic_norms.at[i, 'physicality'] = physicality_df.loc[physicality_df['Word'] == row['word'], 'Average SER'].values[0]
  else:
    psycholinguistic_norms.at[i, 'physicality'] = None

  if row['word'] in glasgow_df['Words'].values:
    psycholinguistic_norms.at[i, 'dataset'] = 'glasgow'
    psycholinguistic_norms.at[i, 'imageability'] = glasgow_df.loc[glasgow_df['Words'] == row['word'], 'IMAG'].values[0]
  else:
    psycholinguistic_norms.at[i, 'imageability'] = None

  if row['word'] in mrc_c[' word'].values:
    psycholinguistic_norms.at[i, 'dataset'] = 'mrc_c'
    psycholinguistic_norms.at[i, 'familiarity'] = mrc_c.loc[mrc_c[' word'] == row['word'], 'mrc.fam'].values[0]
  else:
    psycholinguistic_norms.at[i, 'familiarity'] = None

In [30]:
dataset = physicality_df

nb17_gensim = []
for i, row in dataset.iterrows():
  try:
    nb17_gensim.append(nb_17['/c/en/'+ row['Word']])
  except:
    nb17_gensim.append(None)

dataset['nb17'] = nb17_gensim

w2v_glove = []
for i, row in dataset.iterrows():
  try:
    w2v_glove.append(w2v[row['Word']])
  except:
    w2v_glove.append(None)

dataset['w2v_glove'] = w2v_glove

In [ ]:
dataset=psycholinguistic_norms
nb17_gensim = []
for i, row in dataset.iterrows():
  try:
    nb17_gensim.append(nb_17['/c/en/'+ row['word']])
  except:
    nb17_gensim.append(None)

dataset['nb17'] = nb17_gensim

w2v_glove = []
for i, row in dataset.iterrows():
  try:
    w2v_glove.append(w2v[row['word']])
  except:
    w2v_glove.append(None)

dataset['w2v_glove'] = w2v_glove

In [32]:
#show distributional features oov
w2v_oov = []
for word in df_vocab:
  if word not in w2v_vocab:
    w2v_oov.append(word)
print(len(w2v_oov))

nb_oov = []
for word in df_vocab:
  if word not in nb17_vocab:
    nb_oov.append(word)
print(len(nb_oov))

45
2141


In [33]:
train = physicality_df
test = psycholinguistic_norms[psycholinguistic_norms['physicality'].isna()]
print(len(train))
print(len(test))

max_len = max(len(x) if x is not None else 0 for x in train.nb17)
x_train = np.array([x if x is not None else np.zeros(max_len) for x in train.nb17])
y_train = train['Average SER'].values
max_len = max(len(x) if x is not None else 0 for x in test.nb17)
x_test = np.array([x if x is not None else np.zeros(max_len) for x in test.nb17])

model = SVR(kernel='rbf', C=100, gamma=0.003, epsilon=.1)
model.fit(x_train, y_train)
preds = model.predict(x_test)

new_physicality = np.append(y_train, preds)
print(len(new_physicality))
psycholinguistic_norms['physicality'].update(new_physicality)

5857
1316
7173


C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\1645441730.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  psycholinguistic_norms['physicality'].update(new_physicality)


In [34]:
dataset = glasgow_df

nb17_gensim = []
for i, row in dataset.iterrows():
  try:
    nb17_gensim.append(nb_17['/c/en/'+ row['Words']])
  except:
    nb17_gensim.append(None)

dataset['nb17'] = nb17_gensim

w2v_glove = []
for i, row in dataset.iterrows():
  try:
    w2v_glove.append(w2v[row['Words']])
  except:
    w2v_glove.append(None)

dataset['w2v_glove'] = w2v_glove


In [35]:
train = glasgow_df
test = psycholinguistic_norms[psycholinguistic_norms['imageability'].isna()]
print(len(train))
print(len(test))

max_len = max(len(x) if x is not None else 0 for x in train.nb17)
x_train = np.array([x if x is not None else np.zeros(max_len) for x in train.nb17])
y_train = train['IMAG'].values
x_test = np.array([x if x is not None else np.zeros(max_len) for x in test.nb17])

model = SVR(kernel='rbf', C=100, gamma=0.003, epsilon=.1)
model.fit(x_train, y_train)
preds = model.predict(x_test)

new_df = np.append(y_train, preds)
print(len(new_df))
psycholinguistic_norms['imageability'].update(new_df)

5553
1383
6936


C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3501974845.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  psycholinguistic_norms['imageability'].update(new_df)


In [36]:
dataset = mrc_c

nb17_gensim = []
for i, row in dataset.iterrows():
  try:
    nb17_gensim.append(nb_17['/c/en/'+ row[' word']])
  except:
    nb17_gensim.append(None)

dataset['nb17'] = nb17_gensim

w2v_glove = []
for i, row in dataset.iterrows():
  try:
    w2v_glove.append(w2v[row[' word']])
  except:
    w2v_glove.append(None)

dataset['w2v_glove'] = w2v_glove

In [37]:
train = mrc_c
train = train.dropna(subset=['mrc.fam'])
test = psycholinguistic_norms[psycholinguistic_norms['familiarity'].isna()]

max_len = max(len(x) if x is not None else 0 for x in train.w2v_glove)
x_train = np.array([x if x is not None else np.zeros(max_len) for x in train.w2v_glove])
y_train = train['mrc.fam'].values

x_test = np.array([x if x is not None else np.zeros(max_len) for x in test.w2v_glove])

model = SVR(kernel='rbf', C=100, gamma=0.003, epsilon=.1)
model.fit(x_train, y_train)
preds = model.predict(x_test)

new_df = np.append(y_train, preds)
print(len(new_df))
psycholinguistic_norms['familiarity'].update(new_df)

10697


C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3032782475.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  psycholinguistic_norms['familiarity'].update(new_df)


In [ ]:
psycholinguistic_norms[psycholinguistic_norms['concreteness'].isna()]

,word,concreteness,physicality,imageability,familiarity,dataset,nb17,w2v_glove
0,levels,None,4.0,4.391,629.0,mrc_c,"[0.1153, -0.0045, 0.0382, -0.1849, -0.0703, -0...","[-0.16113281, 0.09472656, 0.15527344, 0.296875..."
3,distributors,None,1.727273,2.516,566.0,mrc_c,"[-0.0031, -0.0357, 0.0107, 0.0051, -0.1192, -0...","[-0.028808594, -0.041015625, -0.04663086, 0.26..."
4,having,None,2.272727,2.571,403.0,mrc_c,"[0.1725, 0.1359, 0.0307, -0.0305, 0.0028, -0.0...","[-0.053955078, 0.05883789, -0.12792969, 0.1245..."
10,efforts,None,4.5,2.84,524.0,mrc_c,"[0.1242, 0.0823, -0.0505, -0.0699, -0.013, -0....","[0.11328125, 0.30664062, -0.10644531, 0.143554..."
14,Job,None,1.909091,4.452,552.0,NaN,None,"[0.20605469, -0.053466797, -0.31835938, -0.265..."
...,...,...,...,...,...,...,...,...
2132,commands,None,4.0,6.194,504.0,mrc_c,"[0.1034, 0.096, -0.0683, -0.1117, -0.054, 0.11...","[0.3203125, 0.111328125, -0.021240234, 0.01226..."
2134,takes,None,2.181818,2.057,526.0,mrc_c,"[0.1648, 0.1312, -0.0016, -0.0455, -0.0783, -0...","[0.12451172, 0.09082031, -0.0048828125, -0.152..."
2137,shocks,None,3.727273,6.441,243.0,mrc_c,"[0.1471, 0.0356, 0.0938, -0.0773, 0.0, -0.0996...","[-0.24902344, 0.37304688, -0.009643555, 0.2148..."
2138,institutions,None,2.909091,3.75,147.0,mrc_c,"[0.0481, 0.0357, -0.0924, -0.2127, -0.0077, 0....","[-0.07763672, 0.13964844, 0.15820312, 0.433593..."


In [38]:
psycholinguistic_norms

,word,concreteness,physicality,imageability,familiarity,dataset,nb17,w2v_glove
0,growing,2.89,4.0,4.391,629.0,mrc_c,"[0.07, -0.0438, 0.1608, 0.0292, 0.045, 0.0157,...","[-0.09423828, 0.33398438, -0.20117188, 0.09521..."
1,legal,1.79,1.272727,5.344,487.0,mrc_c,"[0.0892, 0.057, -0.1395, -0.1324, -0.0377, 0.0...","[-0.23730469, 0.12402344, -0.18457031, -0.0277..."
2,unwillingly,1.62,4.1,3.177,433.0,mrc_c,"[0.1014, 0.0944, -0.0548, 0.0316, 0.143, -0.06...","[0.024047852, -0.05078125, -0.12109375, 0.1757..."
3,finger,5.0,1.727273,2.516,566.0,mrc_c,"[0.1045, -0.0156, 0.1301, 0.0198, -0.0342, 0.0...","[0.25976562, -0.13964844, 0.024414062, -0.1894..."
4,twelve,3.61,2.272727,2.571,403.0,mrc_c,"[0.0126, 0.0423, 0.0044, -0.0022, -0.0625, 0.0...","[-0.026977539, -0.008056641, -0.037841797, 0.2..."
...,...,...,...,...,...,...,...,...
2136,companion,3.75,2.727273,5.219,487.0,mrc_c,"[0.0465, 0.0643, -0.0086, 0.0488, 0.0484, 0.15...","[0.056152344, -0.16113281, -0.24609375, 0.0795..."
2137,materials,None,3.727273,6.441,243.0,mrc_c,"[0.1111, -0.0888, 0.0985, -0.1795, -0.1461, 0....","[0.056396484, 0.18945312, 0.140625, 0.03784179..."
2138,flurry,3.38,2.909091,3.75,147.0,mrc_c,"[0.1584, 0.0739, 0.0866, 0.0534, 0.0503, -0.08...","[0.05859375, 0.14550781, -0.13867188, -0.05151..."
2139,resistance,2.68,3.636364,4.333,443.0,mrc_c,"[0.0892, 0.0049, 0.0443, -0.1899, 0.0239, -0.0...","[0.017578125, 0.07910156, -0.08691406, 0.38476..."


In [ ]:
#download psycholinguistic norms extended to cover the current df vocabulary
psycholinguistic_norms.to_csv('../data/psycholinguistic_norms_EC.tsv', sep='\t')

# Add distributional features to df

In [41]:
df

,Annotation,Word,definitions,BM_original,Source,VVAA,A1,A2,precission_definitions_mean,ic_precission_definitions_mean,familiarity_definitions_mean,imageability_definitions_mean,concreteness_definitions_mean,physicality_definitions_mean,nb17_cosines_definitions_mean,w2v_cosines_definitions_mean
0,control,valuable,something of value,worth a lot of money,https://doi.org/10.1075/celcr.14,0.0,0,0.0,3.00,NaN,573.00,2.93,1.72,3.95,0.246536,0.248655
1,control,valuable,having great material or monetary value especi...,worth a lot of money,https://doi.org/10.1075/celcr.14,1.0,1,1.0,3.90,NaN,504.43,5.12,2.49,3.58,0.218254,0.233507
2,control,valuable,having worth or merit or value,worth a lot of money,https://doi.org/10.1075/celcr.14,0.0,0,0.0,4.00,NaN,475.33,4.30,1.72,3.29,0.398032,0.255454
3,control,roof,a protective covering that covers or forms the...,"the top outer part of a building, temporary st...",https://doi.org/10.1075/celcr.14,1.0,1,1.0,6.33,1.96,504.33,3.89,3.66,2.75,0.208366,0.174519
4,control,roof,protective covering on top of a motor vehicle,"the top outer part of a building, temporary st...",https://doi.org/10.1075/celcr.14,1.0,0,0.0,3.00,2.24,401.40,3.96,3.88,2.27,0.168886,0.164149
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,guided,devious,indirect in departing from the accepted or pro...,The adjective 'devious' means pursuing indirec...,DOI:10.25130/jtuh.28.3.4.2021.25,0.0,NaN,NaN,1.50,NaN,547.71,5.72,2.15,3.12,0.135735,0.107947
876,guided,devious,characterized by insincerity or deceit; evasive,The adjective 'devious' means pursuing indirec...,DOI:10.25130/jtuh.28.3.4.2021.25,0.0,0,0.0,4.00,NaN,552.20,3.93,1.97,3.19,0.327999,0.317092
877,guided,devious,deviating from a straight course,The adjective 'devious' means pursuing indirec...,DOI:10.25130/jtuh.28.3.4.2021.25,1.0,1,1.0,4.50,NaN,549.00,3.71,3.29,2.74,0.119910,0.115508
878,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
wnl = nltk.WordNetLemmatizer()
def process_definitions (x):
  if type(x) == str:
    sentence = word_tokenize(x)
    sentence = [word for word in sentence if word not in stopwords.words('english')]
    sentence = [wnl.lemmatize(word) for word in sentence]
  else:
    sentence = None
  return sentence

In [62]:
df['procesed_definitions'] = df['definitions'].apply(process_definitions)
df = df.dropna(subset=['procesed_definitions'])

In [64]:
nb17_vocab_set = set(nb17_vocab_en)  # Convert nb17_vocab to a set for faster lookups
df['processed_definitions_1'] = df['procesed_definitions'].apply(lambda x: [w for w in x if w in nb17_vocab_set and w.isalpha()])

C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\1261783014.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['processed_definitions_1'] = df['procesed_definitions'].apply(lambda x: [w for w in x if w in nb17_vocab_set and w.isalpha()])


In [65]:
def nb17_cosines_definitions (sentence, target_word):
    sentence = [word for word in sentence if word != target_word]
    if target_word in nb17_vocab_set:
      try:
        return [nb_17.similarity('/c/en/'+ target_word, '/c/en/'+ word) for word in sentence]
      except:
        print(sentence)
    else:
      return None

In [66]:
def w2v_cosines_definitions (sentence, target_word):
  sentence = [word for word in sentence if word != target_word]
  if target_word in w2v_vocab:
    return [w2v.similarity(target_word, word) for word in sentence]
  else:
    return None

In [68]:
df['nb17_cosines_definitions'] = df.apply(lambda row: nb17_cosines_definitions(row['processed_definitions_1'], row['Word']), axis=1)

C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\2343220346.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['nb17_cosines_definitions'] = df.apply(lambda row: nb17_cosines_definitions(row['processed_definitions_1'], row['Word']), axis=1)


In [69]:
w2v_vocab_set = set(w2v_vocab)
df['processed_definitions_1'] = df['procesed_definitions'].apply(lambda x: [w for w in x if w in w2v_vocab_set])
df['w2v_cosines_definitions'] = df.apply(lambda row: w2v_cosines_definitions(row['processed_definitions_1'], row['Word']), axis=1)

C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\1121107641.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['processed_definitions_1'] = df['procesed_definitions'].apply(lambda x: [w for w in x if w in w2v_vocab_set])
C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\1121107641.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['w2v_cosines_definitions'] = df.apply(lambda row: w2v_cosines_definitions(row['processed_definitions_1'], row['Word']), axis=1)


#Add psycholinguistic features to df

In [71]:
psycholinguistic_norms = pd.read_csv('../data/psycholinguistic_norms_EC.tsv', sep='\t')

In [72]:
psycholinguistic_norms_dict = {}
for i, row in psycholinguistic_norms.iterrows():
  psycholinguistic_norms_dict[row['word']] = row

In [73]:
def check_norm (sentence, feature):
  sent_norms = []
  for word in sentence:
    if word in psycholinguistic_norms_dict:
      sent_norms.append(psycholinguistic_norms_dict[word][feature])
    else:
      sent_norms.append(None)
  return sent_norms

In [74]:
#compute psycholinguistic features
for feature in ['familiarity', 'imageability', 'concreteness', 'physicality']:
  definitions = []
  for i, row in df.iterrows():
    definitions.append(check_norm(row['procesed_definitions'], feature))
  print(f'feature {feature} done ')
  df[f'{feature}_definitions'] = definitions

feature familiarity done 
feature imageability done 
feature concreteness done 
feature physicality done 


C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3867787724.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{feature}_definitions'] = definitions
C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3867787724.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{feature}_definitions'] = definitions
C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3867787724.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] =

#Add precission measures to df

In [75]:
def compute_precission (sentence, target_word): #tokenized_sentence
  def_synsets = []
  def_precission = []
  def_ic = []
  for w in sentence:
    if wordnet.synsets(w) and target_word != w:
      try:
        lesk_syn = lesk(sentence, w, synsets=wn.synsets(w))
        lesk_syn_tw = lesk(sentence, target_word, synsets=wn.synsets(target_word))
        def_synsets.append(lesk_syn)
        def_precission.append(lesk_syn.min_depth())
      except:
        def_synsets.append(None)
        def_precission.append(None)
      try:
        def_ic.append(lesk_syn.res_similarity(lesk_syn_tw, brown_ic))
      except:
        def_ic.append(None)
    else:
      def_synsets.append(None)
      def_precission.append(None)
      def_ic.append(None)
  return (def_precission, def_ic)

In [77]:
#compute precission
definition_precission = []
definition_p_ic = []

for i, row in df.iterrows():
  definition_precission.append(compute_precission(word_tokenize(row['definitions']), row['Word'])[0])
  definition_p_ic.append(compute_precission(word_tokenize(row['definitions']), row['Word'])[1])

df['precission_definitions'] = definition_precission
df['ic_precission_definitions'] = definition_p_ic


C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3274743155.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['precission_definitions'] = definition_precission
C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3274743155.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ic_precission_definitions'] = definition_p_ic


#Max min mean

In [78]:
column_list = []
for column in df.columns:
  if column.endswith('_definitions'):
    column_list.append(column)

In [79]:
column_list

['procesed_definitions',
 'nb17_cosines_definitions',
 'w2v_cosines_definitions',
 'familiarity_definitions',
 'imageability_definitions',
 'concreteness_definitions',
 'physicality_definitions',
 'precission_definitions',
 'ic_precission_definitions']

In [80]:
df['nb17_cosines_definitions'] = df['nb17_cosines_definitions'].apply(lambda x: [float(n) for n in x if n is not None] if x is not None else [])
df['w2v_cosines_definitions'] = df['w2v_cosines_definitions'].apply(lambda x: [float(n) for n in x if n is not None] if x is not None else [])

C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3398561607.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['nb17_cosines_definitions'] = df['nb17_cosines_definitions'].apply(lambda x: [float(n) for n in x if n is not None] if x is not None else [])
C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3398561607.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['w2v_cosines_definitions'] = df['w2v_cosines_definitions'].apply(lambda x: [float(n) for n in x if n is not None] if x is not None else []

In [81]:
def mean(x):
  x = [l for l in (x) if type(l) == int or type(l) == float or type(l) == np.float64 or type(l) == np.float32]
  if len(x) == 0:
    return None
  else:
    return sum(x)/len(x)

In [82]:
for column in column_list:
  df[f'{column}_mean'] =  df[column].apply(mean)

C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\266372593.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{column}_mean'] =  df[column].apply(mean)


In [ ]:
for column in column_list:
  print(column)
  max_vals = []
  min_vals = []
  for i, row in df.iterrows():
    a = [l for l in (row[column]) if type(l) == int or type(l) == float or type(l) == np.float64]
    if len(a) == 0:
      max_vals.append(None)
      min_vals.append(None)
    else:
      max_vals.append(max(a))
      min_vals.append(min(a))
  df[f'{column}_max'] = max_vals
  df[f'{column}_min'] = min_vals

#Select best BM definitions and download

In [83]:
#Select best BM: first group all definitions per word
word_index = []
counter = 0
last_word = ''
for i, row in df.iterrows():
  if row['Word'] != last_word:
    last_word = row['Word']
    counter+=1
    word_index.append(counter)
  else:
    word_index.append(counter)
df['word_index'] = word_index

C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3872043202.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['word_index'] = word_index


In [84]:
column_list = ['nb17_cosines_definitions_mean',
 'w2v_cosines_definitions_mean',
 'familiarity_definitions_mean',
 'imageability_definitions_mean',
 'concreteness_definitions_mean',
 'physicality_definitions_mean',
 'precission_definitions_mean',
 'ic_precission_definitions_mean']

In [85]:
df['best_bm'] = 0

C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\3549998930.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['best_bm'] = 0


In [86]:
for i in range (counter)[1:]:
  sup_df = df[df['word_index']==i]
  best_rows = []
  for column in column_list:
    best_rows.append(sup_df[column].idxmax())
  if best_rows:
    try:
      df.loc[statistics.mode(best_rows), 'best_bm'] = 1
    except:
      pass

C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\2430052837.py:5: FutureWarning: The behavior of Series.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  best_rows.append(sup_df[column].idxmax())
C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\2430052837.py:5: FutureWarning: The behavior of Series.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  best_rows.append(sup_df[column].idxmax())
C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\2430052837.py:5: FutureWarning: The behavior of Series.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  best_rows.append(sup_df[column].idxmax())
C:\Users\lucia\AppData\Local\Temp\ipykernel_14500\2430052837.py:5: FutureWarning: The behavior of Series.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this wi

In [87]:
df.groupby('best_bm').size()

best_bm
0.0    789
1.0     90
dtype: int64

In [ ]:
df

In [ ]:
df.to_csv('../EC_with_features.tsv', sep='\t', index=False)